# 05 · Model Evaluation
**Ninja Van Capstone 2026 — Last-Mile Delivery Failure Prediction**

This notebook:
1. Loads all three trained models and applies them to the held-out test set
2. Computes AUC-ROC, Precision, Recall, F1, Brier score + confusion matrices
3. Plots ROC curves, feature importance, and calibration curves
4. Selects the champion model (best F1, AUC-ROC as tiebreaker)
5. Writes `evaluation_report.json` and copies champion model to `champion_model.joblib`

**Outputs:**
- `/content/outputs/evaluation_report.json`
- `/content/outputs/champion_model.joblib`
- `/content/outputs/roc_curves.png`
- `/content/outputs/confusion_matrices.png`
- `/content/outputs/feature_importance.png`
- `/content/outputs/calibration_curves.png`

> **Run order:** Final notebook — run after `04_model_training.ipynb`.


## 1 · Setup & Load Test Data

In [ ]:
import sys, json, joblib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

REPO_DIR = Path("/content/nv-lastmile-training")
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROC_DIR = Path("/content/data/processed")
OUT_DIR  = Path("/content/outputs")

with open(REPO_DIR / "config" / "config.yaml") as f:
    cfg = yaml.safe_load(f)

TARGET_COL = cfg["features"]["target_col"]
THRESHOLD  = 0.5

# Load test split
test_df = pd.read_parquet(PROC_DIR / "features_test.parquet")
X_test_raw = test_df.drop(columns=[TARGET_COL])
y_test     = test_df[TARGET_COL].astype(int).values

# Load feature names
with open(OUT_DIR / "feature_names.json") as f:
    feature_names = json.load(f)

print(f"Test set: {len(test_df):,} rows  |  failure rate: {y_test.mean():.4f}")
print(f"Features: {len(feature_names)}")

In [ ]:
# Load preprocessor and transform test set
preprocessor = joblib.load(OUT_DIR / "preprocessor.joblib")
X_test = preprocessor.transform(X_test_raw)
print(f"Transformed test set shape: {X_test.shape}")

## 2 · Load All Trained Models

In [ ]:
models = {}
for name in ["lgbm", "rf", "lr"]:
    path = OUT_DIR / f"model_{name}.joblib"
    if path.exists():
        models[name] = joblib.load(path)
        print(f"✅ Loaded {name:5s} → {path.name}")
    else:
        print(f"❌ {path} not found — skipping")

if not models:
    raise FileNotFoundError(
        "No trained models found in outputs/. Run 04_model_training.ipynb first."
    )

## 3 · Evaluate Each Model on Test Set

In [ ]:
from evaluate import evaluate_model

results = {}
for name, model in models.items():
    print(f"\n{'='*55}")
    print(f"  Evaluating: {name.upper()}")
    results[name] = evaluate_model(
        model, X_test, y_test,
        model_name=name,
        threshold=THRESHOLD,
        feature_names=feature_names,
    )

## 4 · ROC Curves

In [ ]:
from evaluate import plot_roc_curves

roc_path = str(OUT_DIR / "roc_curves.png")
plot_roc_curves(models, results, y_test, save_path=roc_path)

from IPython.display import Image
Image(roc_path)

## 5 · Confusion Matrices

In [ ]:
from evaluate import plot_confusion_matrices

cm_path = str(OUT_DIR / "confusion_matrices.png")
plot_confusion_matrices(results, save_path=cm_path)

Image(cm_path)

## 6 · LightGBM Feature Importance

In [ ]:
from evaluate import plot_feature_importance

if "lgbm" in models:
    fi_path = str(OUT_DIR / "feature_importance.png")
    plot_feature_importance(
        models["lgbm"],
        feature_names=feature_names,
        save_path=fi_path,
        top_n=20,
    )
    Image(fi_path)
else:
    print("⚠️  LightGBM model not available — skipping feature importance plot")

## 7 · Calibration Curves

In [ ]:
from evaluate import plot_calibration_curves

cal_path = str(OUT_DIR / "calibration_curves.png")
plot_calibration_curves(results, y_test, save_path=cal_path)

Image(cal_path)

## 8 · Metric Summary Table

In [ ]:
summary_rows = []
for name, res in results.items():
    summary_rows.append({
        "Model":     name.upper(),
        "AUC-ROC":   f"{res['auc_roc']:.4f}",
        "Precision": f"{res['precision']:.4f}",
        "Recall":    f"{res['recall']:.4f}",
        "F1":        f"{res['f1']:.4f}",
        "Brier":     f"{res['brier']:.4f}",
    })

summary_df = pd.DataFrame(summary_rows).set_index("Model")
print("\n" + "=" * 65)
print("  TEST SET EVALUATION SUMMARY")
print("=" * 65)
print(summary_df.to_string())
print("=" * 65)

## 9 · Select Champion Model

In [ ]:
from evaluate import select_champion

champion = select_champion(results)
print(f"\n🏆 Champion model: {champion.upper()}")

## 10 · Write Evaluation Report & Copy Champion

In [ ]:
from evaluate import write_evaluation_report, copy_champion_model

report = write_evaluation_report(
    results=results,
    champion=champion,
    feature_names=feature_names,
    test_set_size=len(y_test),
    failure_rate_test=float(y_test.mean()),
    save_path=str(OUT_DIR / "evaluation_report.json"),
    threshold=THRESHOLD,
)

champion_path = copy_champion_model(champion, str(OUT_DIR))

print("\n📋 Evaluation Report:")
print(json.dumps(report, indent=2, default=float))

## 11 · Download Artifacts from Colab

Run this cell to download all model artifacts to your local machine.

In [ ]:
from google.colab import files

artifacts = [
    OUT_DIR / "champion_model.joblib",
    OUT_DIR / "preprocessor.joblib",
    OUT_DIR / "model_lgbm.joblib",
    OUT_DIR / "model_rf.joblib",
    OUT_DIR / "model_lr.joblib",
    OUT_DIR / "feature_names.json",
    OUT_DIR / "evaluation_report.json",
]

print("⬇️  Downloading artifacts...")
for path in artifacts:
    if path.exists():
        files.download(str(path))
        print(f"   Downloaded: {path.name}")
    else:
        print(f"   ⚠️  Not found: {path.name}")

In [ ]:
print("=" * 70)
print("  NOTEBOOK 05 COMPLETE — TRAINING PIPELINE FINISHED")
print("=" * 70)
print()
print("  Artifacts produced:")
print(f"    champion_model.joblib  → {OUT_DIR / 'champion_model.joblib'}")
print(f"    preprocessor.joblib   → {OUT_DIR / 'preprocessor.joblib'}")
print(f"    model_lgbm.joblib     → {OUT_DIR / 'model_lgbm.joblib'}")
print(f"    model_rf.joblib       → {OUT_DIR / 'model_rf.joblib'}")
print(f"    model_lr.joblib       → {OUT_DIR / 'model_lr.joblib'}")
print(f"    feature_names.json    → {OUT_DIR / 'feature_names.json'}")
print(f"    evaluation_report.json→ {OUT_DIR / 'evaluation_report.json'}")
print()
print("  Next step: Copy joblib files to nv-lastmile-demo/models/")
print("             to power the Gradio POC app.")
print("=" * 70)